# Exploratory Data Analysis - 20 Newsgroups Dataset

This notebook performs comprehensive exploratory data analysis on the 20 Newsgroups dataset.

**Objective:** Understand the dataset structure, distribution, and characteristics before model development.

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path

from src.data_loader import DataLoader
from src.preprocessing import TextPreprocessor

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Create results directory
Path('../results/figures').mkdir(parents=True, exist_ok=True)
Path('../results/metrics').mkdir(parents=True, exist_ok=True)
Path('../results/reports').mkdir(parents=True, exist_ok=True)

## 1. Load Dataset

In [ ]:
# Load configuration
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Initialize data loader
loader = DataLoader('../config/config.yaml')

# Load dataset (downloads if not cached)
dataset = loader.load_dataset()

# Create DataFrame
df = loader.create_dataframe(dataset)

print(f"Dataset loaded successfully!")
print(f"Total documents: {len(df):,}")
print(f"Categories: {df['category'].nunique()}")
print(f"Splits: {df['dataset_split'].value_counts().to_dict()}")

## 2. Dataset Overview

In [ ]:
# Basic info
print("=== DATASET INFO ===")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

# Check for empty documents
empty_count = (df['document_text'].str.strip() == '').sum()
print(f"Empty documents: {empty_count}")

# Check for duplicates
dup_count = df['document_text'].duplicated().sum()
print(f"Duplicate documents: {dup_count}")

## 3. Category Distribution

In [ ]:
# Category distribution
cat_dist = df['category'].value_counts().sort_values(ascending=False)
cat_df = pd.DataFrame({'category': cat_dist.index, 'count': cat_dist.values})

print("=== CATEGORY DISTRIBUTION ===")
print(cat_df.to_string(index=False))

# Plot category distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Horizontal bar chart
cat_dist_asc = cat_dist.sort_values(ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(cat_dist_asc)))
bars = axes[0].barh(range(len(cat_dist_asc)), cat_dist_asc.values, color=colors)
axes[0].set_yticks(range(len(cat_dist_asc)))
axes[0].set_yticklabels(cat_dist_asc.index, fontsize=10)
axes[0].set_xlabel('Number of Documents', fontsize=12)
axes[0].set_title('Documents per Category', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, val in zip(bars, cat_dist_asc.values):
    axes[0].text(val + 50, bar.get_y() + bar.get_height()/2, f'{val:,}',
                va='center', fontsize=9, fontweight='bold')

# Pie chart for top 10
top10 = cat_dist.head(10)
others = pd.Series({'Others': cat_dist.iloc[10:].sum()})
pie_data = pd.concat([top10, others])
axes[1].pie(pie_data.values, labels=pie_data.index, autopct='%1.1f%%',
           colors=plt.cm.Set3(np.linspace(0, 1, len(pie_data))),
           startangle=90)
axes[1].set_title('Top 10 Categories + Others', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/category_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 4. Document Length Analysis

In [ ]:
# Document length statistics
df['doc_length'] = df['document_text'].str.len()
df['word_count'] = df['document_text'].str.split().str.len()

print("=== DOCUMENT LENGTH STATISTICS ===")
print(f"Character length:\n{df['doc_length'].describe()}")
print(f"\nWord count:\n{df['word_count'].describe()}")

# By split
print("\n=== BY SPLIT ===")
for split in ['train', 'test']:
    subset = df[df['dataset_split'] == split]
    print(f"\n{split.upper()}:")
    print(f"  Mean chars: {subset['doc_length'].mean():.0f}")
    print(f"  Mean words: {subset['word_count'].mean():.0f}")
    print(f"  Median chars: {subset['doc_length'].median():.0f}")

In [ ]:
# Document length visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Histogram of document lengths (characters)
axes[0, 0].hist(df['doc_length'], bins=50, color='#2E86C1', alpha=0.8, edgecolor='white')
axes[0, 0].axvline(df['doc_length'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["doc_length"].mean():.0f}')
axes[0, 0].axvline(df['doc_length'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["doc_length"].median():.0f}')
axes[0, 0].set_xlabel('Document Length (characters)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Document Length Distribution (Characters)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Histogram of word counts
axes[0, 1].hist(df['word_count'], bins=50, color='#27AE60', alpha=0.8, edgecolor='white')
axes[0, 1].axvline(df['word_count'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["word_count"].mean():.0f}')
axes[0, 1].axvline(df['word_count'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["word_count"].median():.0f}')
axes[0, 1].set_xlabel('Word Count')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Document Length Distribution (Words)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Box plot by category
cat_order = df.groupby('category')['doc_length'].median().sort_values().index
sns.boxplot(data=df, y='category', x='doc_length', order=cat_order, ax=axes[1, 0], palette='viridis')
axes[1, 0].set_xlabel('Document Length (characters)')
axes[1, 0].set_ylabel('Category')
axes[1, 0].set_title('Document Length by Category')
axes[1, 0].grid(True, alpha=0.3, axis='x')

# Box plot by split
sns.boxplot(data=df, x='dataset_split', y='doc_length', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_xlabel('Dataset Split')
axes[1, 1].set_ylabel('Document Length (characters)')
axes[1, 1].set_title('Document Length by Split')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/figures/document_length_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 5. Text Content Analysis

In [ ]:
# Sample documents from different categories
print("=== SAMPLE DOCUMENTS ===")
for cat in sorted(df['category'].unique())[:5]:
    sample = df[df['category'] == cat].iloc[0]
    print(f"\n--- {cat} ---")
    print(sample['document_text'][:500])
    print("...")

In [ ]:
# Word frequency analysis (on a sample for speed)
from collections import Counter
import re

# Sample 1000 documents for word frequency
sample_df = df.sample(n=min(1000, len(df)), random_state=42)

all_words = []
for text in sample_df['document_text']:
    # Simple tokenization
    words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())
    all_words.extend(words)

word_freq = Counter(all_words)
top_words = word_freq.most_common(30)

print("=== TOP 30 MOST FREQUENT WORDS (sample of 1000 docs) ===")
for word, count in top_words:
    print(f"  {word}: {count:,}")

In [ ]:
# Word frequency visualization
top_words_df = pd.DataFrame(top_words, columns=['word', 'frequency'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 20 words
top20 = top_words_df.head(20).sort_values('frequency')
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top20)))
bars = axes[0].barh(range(len(top20)), top20['frequency'], color=colors)
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels(top20['word'], fontsize=10)
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 20 Most Frequent Words')
axes[0].grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, top20['frequency']):
    axes[0].text(val + 5, bar.get_y() + bar.get_height()/2, f'{val:,}',
                va='center', fontsize=9)

# Zipf's law plot
ranks = np.arange(1, len(top_words_df) + 1)
freqs = top_words_df['frequency'].values
axes[1].loglog(ranks, freqs, 'o-', color='#E74C3C', alpha=0.7)
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Frequency')
axes[1].set_title("Zipf's Law: Word Frequency vs Rank")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/word_frequency_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 6. Class Balance Analysis

In [ ]:
# Class balance analysis
cat_counts = df['category'].value_counts()

balance_stats = {
    'total_categories': len(cat_counts),
    'min_count': cat_counts.min(),
    'max_count': cat_counts.max(),
    'mean_count': cat_counts.mean(),
    'std_count': cat_counts.std(),
    'imbalance_ratio': cat_counts.max() / cat_counts.min(),
    'cv': cat_counts.std() / cat_counts.mean()  # Coefficient of variation
}

print("=== CLASS BALANCE STATISTICS ===")
for key, value in balance_stats.items():
    print(f"  {key}: {value:.2f}" if isinstance(value, float) else f"  {key}: {value}")

# Visualize class balance
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(range(len(cat_counts)), cat_counts.sort_values().values, color='#2E86C1', alpha=0.8)
axes[0].axhline(cat_counts.mean(), color='red', linestyle='--', label=f'Mean: {cat_counts.mean():.0f}')
axes[0].set_xlabel('Category (sorted by count)')
axes[0].set_ylabel('Document Count')
axes[0].set_title('Class Distribution Balance')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Cumulative distribution
cumsum = cat_counts.sort_values(ascending=False).cumsum()
cumsum_pct = cumsum / cumsum.iloc[-1] * 100
axes[1].plot(range(1, len(cumsum) + 1), cumsum_pct, 'o-', color='#E74C3C', linewidth=2)
axes[1].axhline(80, color='gray', linestyle='--', label='80%')
axes[1].axhline(90, color='gray', linestyle=':', label='90%')
axes[1].set_xlabel('Number of Categories (top-k)')
axes[1].set_ylabel('Cumulative % of Documents')
axes[1].set_title('Cumulative Category Coverage')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/class_balance_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 7. Split Distribution

In [ ]:
# Split distribution analysis
split_dist = df['dataset_split'].value_counts()
print("=== SPLIT DISTRIBUTION ===")
print(split_dist)

# Category distribution within each split
split_cat = pd.crosstab(df['category'], df['dataset_split'])
print("\n=== CATEGORY × SPLIT ===")
print(split_cat)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Split counts
colors = ['#2E86C1', '#27AE60', '#E74C3C']
split_dist.plot(kind='bar', ax=axes[0], color=colors[:len(split_dist)])
axes[0].set_title('Dataset Split Distribution')
axes[0].set_xlabel('Split')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(split_dist):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Stacked bar by category
split_cat.plot(kind='bar', stacked=True, ax=axes[1], colormap='viridis')
axes[1].set_title('Category Distribution by Split')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Split')

plt.tight_layout()
plt.savefig('../results/figures/split_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 8. Category-wise Statistics Summary

In [ ]:
# Comprehensive category statistics
cat_stats = df.groupby('category').agg({
    'doc_length': ['count', 'mean', 'median', 'std', 'min', 'max'],
    'word_count': ['mean', 'median']
}).round(2)

cat_stats.columns = ['_'.join(col).strip() for col in cat_stats.columns.values]
cat_stats = cat_stats.rename(columns={'doc_length_count': 'count'})
cat_stats = cat_stats.sort_values('count', ascending=False)

print(cat_stats.to_string())

# Save to CSV
cat_stats.to_csv('../results/reports/category_statistics.csv')
print("\nSaved to ../results/reports/category_statistics.csv")

## 9. Observations and Conclusions

### Key Observations:

1. **Dataset Size**: ~18,846 documents across 20 categories
2. **Class Balance**: Relatively balanced with imbalance ratio ~1.5-2.0
3. **Document Length**: High variance - some very short, some very long documents
4. **Headers/Footers**: Raw documents contain newsgroup headers, signatures, quoted text
5. **Split Distribution**: Original train/test split preserved, validation split from training

### Preprocessing Implications:

- **Header removal is critical** - headers contain newsgroup names that would leak labels
- **Quoted text removal** - quoted portions may contain text from other categories
- **Length normalization** - consider truncation or handling of very long documents
- **TF-IDF parameters** - max_features=15000, ngram_range=(1,2) seems reasonable

### Model Design Considerations:

- MLP with 2 hidden layers (256, 128) should be sufficient
- Dropout (0.4, 0.3) for regularization
- Early stopping to prevent overfitting
- Class weights not needed due to reasonable balance

In [ ]:
# Save EDA summary
eda_summary = {
    'total_documents': int(len(df)),
    'num_categories': int(df['category'].nunique()),
    'split_distribution': df['dataset_split'].value_counts().to_dict(),
    'category_distribution': df['category'].value_counts().to_dict(),
    'empty_documents': int((df['document_text'].str.strip() == '').sum()),
    'duplicate_documents': int(df['document_text'].duplicated().sum()),
    'avg_doc_length_chars': float(df['doc_length'].mean()),
    'median_doc_length_chars': float(df['doc_length'].median()),
    'avg_word_count': float(df['word_count'].mean()),
    'class_imbalance_ratio': float(df['category'].value_counts().max() / df['category'].value_counts().min()),
    'observations': [
        "Dataset is relatively balanced across 20 categories",
        "Raw documents contain headers, footers, and quoted text requiring cleaning",
        "Document lengths vary significantly (high variance)",
        "Original train/test split preserved with stratified validation split"
    ]
}

import json
with open('../results/metrics/eda_summary.json', 'w') as f:
    json.dump(eda_summary, f, indent=2)

print("EDA summary saved to ../results/metrics/eda_summary.json")
print("\n=== EDA COMPLETE ===")